## Train the scientific baseline (Wav2Vec2)

Uses the manifests from `01_finalize_validated_dataset.ipynb`: each row is a **30 s** clip with `start_sec` and `clip_sec`. We fine-tune `facebook/wav2vec2-base` for binary labels, with **SpecAugment turned off** so this run stays the baseline for later experiments.

Checkpoints: `./scientific_baseline_wav2vec2`. Figures: `scientific_baseline_training_curves.png`, `scientific_baseline_confusion_matrix.png`. Optional Hub repo: `Mrsmetamorphosis/dementia-wav2vec-scientific-baseline`.

Mount Drive, then `cd` into this repository (same pattern as the dataset notebook).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/NeuralNets_Project_DementiaNet

Imports and seeds.

In [ ]:
import torch
import torchaudio
import pandas as pd
import numpy as np
import random

from datasets import Dataset
from transformers import (
    AutoConfig,
    Wav2Vec2Processor,
    Wav2Vec2ForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

from config import *

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)


Load manifests (tab-separated).

In [ ]:
train_df = pd.read_csv(TRAIN_MANIFEST, sep="\t")
valid_df = pd.read_csv(VALID_MANIFEST, sep="\t")
test_df = pd.read_csv(TEST_MANIFEST, sep="\t")

print("Train clips:", len(train_df))
print("Valid clips:", len(valid_df))
print("Test clips:", len(test_df))


Turn string labels into 0/1 integers.

In [ ]:
label_map = {"nodementia": 0, "dementia": 1}
train_df["label"] = train_df["label"].map(label_map)
valid_df["label"] = valid_df["label"].map(label_map)
test_df["label"] = test_df["label"].map(label_map)
train_df.head()


Load the waveform slice for each row. Native sample rate comes from the file; we resample to 16 kHz if needed so the processor always sees the same rate.

In [ ]:
def load_audio_segment(row):
    path = row["path"]
    target_sr = 16000
    start = float(row["start_sec"])
    clip = float(row["clip_sec"])
    info = torchaudio.info(path)
    sr = info.sample_rate
    frame_offset = int(start * sr)
    num_frames = int(clip * sr)
    waveform, sr = torchaudio.load(path, frame_offset=frame_offset, num_frames=num_frames)
    if sr != target_sr:
        waveform = torchaudio.functional.resample(waveform, sr, target_sr)
        sr = target_sr
    return waveform.squeeze().numpy()


Hugging Face `Dataset` objects for each split.

In [ ]:
train_dataset = Dataset.from_pandas(train_df)
valid_dataset = Dataset.from_pandas(valid_df)
test_dataset = Dataset.from_pandas(test_df)


Uncomment if you push to Hub or pull a private checkpoint.

In [ ]:
from huggingface_hub import login

# login()


Feature extractor front-end.

In [ ]:
processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base")

Preprocessing map: raw audio to `input_values`.

In [ ]:
def preprocess(batch):
    audio = load_audio_segment(batch)
    inputs = processor(
        audio,
        sampling_rate=16000,
        return_tensors="pt",
        padding=True,
    )
    batch["input_values"] = inputs.input_values[0]
    batch["labels"] = batch["label"]
    return batch


train_dataset = train_dataset.map(preprocess)
valid_dataset = valid_dataset.map(preprocess)
test_dataset = test_dataset.map(preprocess)


Binary head on Wav2Vec2; `apply_spec_augment` is False for this baseline.

In [ ]:
label2id = {"nodementia": 0, "dementia": 1}
id2label = {0: "nodementia", 1: "dementia"}

config = AutoConfig.from_pretrained("facebook/wav2vec2-base")
config.num_labels = 2
config.label2id = label2id
config.id2label = id2label
config.attention_dropout = 0.1
config.hidden_dropout = 0.1
config.feat_proj_dropout = 0.1
config.final_dropout = 0.2
config.apply_spec_augment = False

model = Wav2Vec2ForSequenceClassification.from_pretrained(
    "facebook/wav2vec2-base",
    config=config,
    ignore_mismatched_sizes=True,
)
model.freeze_feature_encoder()


Balanced class weights for the loss (same idea as the SpecAugment notebook).

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

labels = train_dataset["labels"]
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(labels),
    y=labels,
)
class_weights = torch.tensor(class_weights, dtype=torch.float)
print("class_weights:", class_weights)


Training loop settings (tweak batch size / epochs here).

In [ ]:
training_args = TrainingArguments(
    output_dir="./scientific_baseline_wav2vec2",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=22,
    logging_steps=10,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    load_best_model_at_end=True,
    push_to_hub=True,
    hub_model_id="Mrsmetamorphosis/dementia-wav2vec-scientific-baseline",
)


Trainer subclass: weighted cross-entropy.

In [ ]:
import torch.nn as nn


class WeightedTrainer(Trainer):

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        w = class_weights.to(model.device)
        loss_fct = nn.CrossEntropyLoss(weight=w)
        loss = loss_fct(
            logits.view(-1, model.config.num_labels),
            labels.view(-1),
        )
        return (loss, outputs) if return_outputs else loss


Metrics reported each epoch (macro F1 drives checkpoint selection).

In [ ]:
def compute_metrics(pred):
    preds = np.argmax(pred.predictions, axis=1)
    labels = pred.label_ids
    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )
    _, _, f1_weighted, _ = precision_recall_fscore_support(
        labels, preds, average="weighted", zero_division=0
    )
    _, _, f1_class, _ = precision_recall_fscore_support(
        labels, preds, average=None, zero_division=0
    )
    acc = accuracy_score(labels, preds)
    return {
        "accuracy": acc,
        "f1_macro": f1_macro,
        "f1_weighted": f1_weighted,
        "precision_macro": precision_macro,
        "recall_macro": recall_macro,
        "f1_nodementia": f1_class[0],
        "f1_dementia": f1_class[1],
    }


Pad waveforms to the longest item in each batch.

In [ ]:
def data_collator(features):
    input_features = [{"input_values": f["input_values"]} for f in features]
    label_features = [f["labels"] for f in features]
    batch = processor.pad(input_features, padding=True, return_tensors="pt")
    batch["labels"] = torch.tensor(label_features, dtype=torch.long)
    return batch


Wire up the trainer.

In [ ]:
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)


Train (this is the slow part).

In [ ]:
trainer.train()

Evaluate on the held-out test manifest.

In [ ]:
test_pred = trainer.predict(test_dataset)
print(test_pred.metrics)
_, counts = np.unique(np.argmax(test_pred.predictions, axis=1), return_counts=True)
print("Predicted label counts:", counts)


Write JSON for `notebooks/specaugment/03_compare_baseline_vs_specaugment.ipynb`.

In [ ]:
import json
from pathlib import Path

out = Path("results") / "baseline_test_metrics.json"
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(
    json.dumps({k: float(v) for k, v in test_pred.metrics.items()}, indent=2),
    encoding="utf-8",
)
print("Wrote", out.resolve())


Loss / macro-F1 curves from the trainer log.

In [ ]:
import matplotlib.pyplot as plt

logs = pd.DataFrame(trainer.state.log_history)
logs = logs[logs["epoch"].notna()]
train_loss = logs[logs["loss"].notna()][["epoch", "loss"]]
val_loss = logs[logs["eval_loss"].notna()][["epoch", "eval_loss"]]
val_f1 = logs[logs["eval_f1_macro"].notna()][["epoch", "eval_f1_macro"]]

plt.figure(figsize=(9, 4))
plt.plot(train_loss["epoch"], train_loss["loss"], label="train loss")
plt.plot(val_loss["epoch"], val_loss["eval_loss"], label="valid loss")
plt.plot(val_f1["epoch"], val_f1["eval_f1_macro"], label="valid macro F1")
plt.xlabel("epoch")
plt.ylabel("value")
plt.title("Scientific baseline — training curves")
plt.legend()
plt.tight_layout()
plt.savefig("scientific_baseline_training_curves.png", dpi=200, bbox_inches="tight")
plt.show()


Macro precision / recall / F1 on the test split.

In [ ]:
pred_labels = test_pred.label_ids
pred_classes = np.argmax(test_pred.predictions, axis=1)
precision, recall, f1, _ = precision_recall_fscore_support(
    pred_labels, pred_classes, average="macro", zero_division=0
)
accuracy = accuracy_score(pred_labels, pred_classes)
print("accuracy:", accuracy)
print("macro P / R / F1:", precision, recall, f1)


Confusion matrix on test clips.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

pred_classes = np.argmax(test_pred.predictions, axis=1)
labels_cm = test_pred.label_ids
cm = confusion_matrix(labels_cm, pred_classes)
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["No dementia", "Dementia"],
)
disp.plot(cmap="Blues")
plt.title("Scientific baseline — confusion matrix (test)")
plt.tight_layout()
plt.savefig("scientific_baseline_confusion_matrix.png", dpi=200, bbox_inches="tight")
plt.show()


Optional: push weights and processor to the Hub.

In [ ]:
trainer.push_to_hub()

In [ ]:
processor.push_to_hub("Mrsmetamorphosis/dementia-wav2vec-scientific-baseline")